# Task 1 — Classical Model Optimization

This notebook starts directly from **Task 1** and is designed to run **without relying on earlier baseline cells**.

It follows the same overall flow used in the deep-model TinyML notebook:
- standalone configuration and dataset loading
- saved-model loading from the registry
- baseline metrics on fixed hold-out splits
- one technique at a time
- inside each technique, dataset-by-dataset execution
- final combined plots

The techniques here are **classical-model lightweight optimization techniques**, not canonical neural TinyML compression techniques.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import json
import time
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import xgboost as xgb
import lightgbm as lgb

sns.set_style("whitegrid")

## Task 1 Configuration

In [ ]:
# Dataset paths
# Update these if your local paths are different.

CICEVSE_PATH = r'C:\Users\100987869\Desktop\EV-Survey-Models2\CICEVSE2024\Subsets\CICEVSE2024_12classes_kmeans_rule_sampled1000.csv'
CICIDS_PATH  = r'c:\Users\100987869\Downloads\cic_0.01km.csv'
NFTON_PATH   = r'C:\Users\100987869\Desktop\EV-Survey-Models2\NF-Ton-IoT-V2\NF-ToN-IoT-V2.parquet'

# Saved model registry paths
MODEL_DIR   = 'models'
MODELS_JSON = 'models.json'

CLASSICAL_MODEL_NAMES = [
    'Logistic Regression',
    'SVM',
    'KNN',
]

TEST_SIZE = 0.20
RANDOM_STATE = 42
NFTON_SUBSET_SIZE = 20000

print('Classical optimization configuration loaded.')

## Registry Helpers

In [ ]:
def _load_registry() -> dict:
    if os.path.exists(MODELS_JSON):
        with open(MODELS_JSON, 'r') as f:
            return json.load(f)
    return {}

def _require_file(path_str: str, label: str):
    if not os.path.exists(path_str):
        raise FileNotFoundError(f'{label} not found: {path_str}')

def _normalise_path(path_str: str) -> str:
    return str(Path(path_str))

def _get_model_entry(dataset_tag: str, model_name: str):
    registry = _load_registry()
    exact_key = f'[{dataset_tag}] {model_name}'
    if exact_key in registry:
        return exact_key, registry[exact_key]

    normalized_exact = exact_key.strip().lower()
    for key, value in registry.items():
        if key.strip().lower() == normalized_exact:
            return key, value

    for key, value in registry.items():
        key_l = key.lower()
        if f'[{dataset_tag.lower()}]' in key_l and model_name.lower() in key_l:
            return key, value

    raise KeyError(f'Model entry not found in registry: {exact_key}')

def _load_saved_sklearn_model(dataset_tag: str, model_name: str):
    key, entry = _get_model_entry(dataset_tag, model_name)
    model_path = _normalise_path(entry['model_path'])
    _require_file(model_path, key)
    model = joblib.load(model_path)
    return model, key, entry

## Dataset Loading and Preprocessing

In [ ]:
# CICEVSE2024
_require_file(CICEVSE_PATH, 'CICEVSE2024 dataset')
df_cicevse_2024 = pd.read_csv(CICEVSE_PATH)

labelencoder_cicevse = LabelEncoder()
df_cicevse_2024.iloc[:, -1] = labelencoder_cicevse.fit_transform(df_cicevse_2024.iloc[:, -1])

if df_cicevse_2024.isnull().values.any() or np.isinf(df_cicevse_2024.select_dtypes(include=[np.number])).values.any():
    df_cicevse_2024.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cicevse_2024.fillna(0, inplace=True)

X_cicevse = df_cicevse_2024.drop(['Label'], axis=1)
y_cicevse = df_cicevse_2024.iloc[:, -1]

print('CICEVSE2024 shape:', X_cicevse.shape)
print('CICEVSE2024 classes:', len(np.unique(y_cicevse)))

# CICIDS2017
_require_file(CICIDS_PATH, 'CICIDS2017 dataset')
df_cicids = pd.read_csv(CICIDS_PATH)

labelencoder_cicids = LabelEncoder()
df_cicids.iloc[:, -1] = labelencoder_cicids.fit_transform(df_cicids.iloc[:, -1])

if df_cicids.isnull().values.any() or np.isinf(df_cicids.select_dtypes(include=[np.number])).values.any():
    df_cicids.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cicids.fillna(0, inplace=True)

X_cicids = df_cicids.iloc[:, :-1]
y_cicids = df_cicids.iloc[:, -1]

print('CICIDS2017 shape:', X_cicids.shape)
print('CICIDS2017 classes:', len(np.unique(y_cicids)))

# NF-ToN-IoT-V2
_require_file(NFTON_PATH, 'NF-ToN-IoT-V2 dataset')
df_nfton = pd.read_parquet(NFTON_PATH)

label_col_candidates = ['Attack', 'Label', 'label']
label_col_nfton = None
for col in label_col_candidates:
    if col in df_nfton.columns:
        label_col_nfton = col
        break

if label_col_nfton is None:
    label_col_nfton = df_nfton.columns[-1]

df_nfton = df_nfton.copy()
for col in df_nfton.columns:
    if df_nfton[col].dtype == 'object':
        if col == label_col_nfton:
            continue
        df_nfton[col] = df_nfton[col].astype('category').cat.codes

labelencoder_nfton = LabelEncoder()
df_nfton[label_col_nfton] = labelencoder_nfton.fit_transform(df_nfton[label_col_nfton])

if df_nfton.isnull().values.any() or np.isinf(df_nfton.select_dtypes(include=[np.number])).values.any():
    df_nfton.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_nfton.fillna(0, inplace=True)

X_nfton = df_nfton.drop(columns=[label_col_nfton])
y_nfton = df_nfton[label_col_nfton]

if len(X_nfton) > NFTON_SUBSET_SIZE:
    X_nfton_sub, _, y_nfton_sub, _ = train_test_split(
        X_nfton, y_nfton,
        train_size=NFTON_SUBSET_SIZE,
        stratify=y_nfton,
        random_state=RANDOM_STATE
    )
else:
    X_nfton_sub, y_nfton_sub = X_nfton, y_nfton

print('NF-ToN-IoT-V2 original shape:', X_nfton.shape)
print('NF-ToN-IoT-V2 subset shape:', X_nfton_sub.shape)
print('NF-ToN-IoT-V2 classes:', len(np.unique(y_nfton_sub)))

## Fixed Hold-Out Splits

In [ ]:
# Task uses fixed hold-out splits.
# Not finished due fold-specific baseline checkpoints and test splits not being available inside this standalone notebook.

X_train_cicevse, X_test_cicevse, y_train_cicevse, y_test_cicevse = train_test_split(
    X_cicevse, y_cicevse, test_size=TEST_SIZE, stratify=y_cicevse, random_state=RANDOM_STATE
)

X_train_cicids, X_test_cicids, y_train_cicids, y_test_cicids = train_test_split(
    X_cicids, y_cicids, test_size=TEST_SIZE, stratify=y_cicids, random_state=RANDOM_STATE
)

X_train_nfton, X_test_nfton, y_train_nfton, y_test_nfton = train_test_split(
    X_nfton_sub, y_nfton_sub, test_size=TEST_SIZE, stratify=y_nfton_sub, random_state=RANDOM_STATE
)

print('Hold-out splits prepared.')

## Classical Model Loading

In [ ]:
# CICEVSE2024 classical models
lr_cicevse_model, lr_cicevse_key, lr_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'Logistic Regression')
svm_cicevse_model, svm_cicevse_key, svm_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'SVM')
knn_cicevse_model, knn_cicevse_key, knn_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'KNN')
print('Loaded CICEVSE2024 classical models.')

# CICIDS2017 classical models
lr_cicids_model, lr_cicids_key, lr_cicids_entry = _load_saved_sklearn_model('CICIDS', 'Logistic Regression')
svm_cicids_model, svm_cicids_key, svm_cicids_entry = _load_saved_sklearn_model('CICIDS', 'SVM')
knn_cicids_model, knn_cicids_key, knn_cicids_entry = _load_saved_sklearn_model('CICIDS', 'KNN')
print('Loaded CICIDS2017 classical models.')

# NF-ToN-IoT-V2 classical models
lr_nfton_model, lr_nfton_key, lr_nfton_entry = _load_saved_sklearn_model('NFToN', 'Logistic Regression')
svm_nfton_model, svm_nfton_key, svm_nfton_entry = _load_saved_sklearn_model('NFToN', 'SVM')
knn_nfton_model, knn_nfton_key, knn_nfton_entry = _load_saved_sklearn_model('NFToN', 'KNN')
print('Loaded NF-ToN-IoT-V2 classical models.')

## Technique Reference Table

The table below gives a short plain-language description of each classical-model optimization technique and the foundational paper used as the reference point for the idea.

| Technique | Plain explanation | Foundational reference |
|---|---|---|
| Feature Selection | Keep only the most informative input variables so the model is smaller and faster. | Guyon and Elisseeff, *An Introduction to Variable and Feature Selection* (JMLR 2003) |
| PCA | Replace many original features with a smaller set of components that captures most of the variation. | Pearson, *On Lines and Planes of Closest Fit to Systems of Points in Space* (1901) |
| L1 Sparsification | Use an L1 penalty so some coefficients go exactly to zero, which makes the model simpler. | Tibshirani, *Regression Shrinkage and Selection via the Lasso* (1996) |
| Prototype Reduction | Keep only a reduced set of training instances for nearest-neighbor classification. | Hart, *The Condensed Nearest Neighbor Rule* (1968) |

**Important note.** Not every technique is defined for every classical model. Unsupported combinations are marked with a normal-text note instead of forcing a fake implementation.

## Optimization Helpers

In [ ]:
def get_model_family(model_name: str) -> str:
    tree_names = ['Decision Tree', 'Random Forest', 'Extra Trees', 'XGBoost', 'LightGBM']
    return 'Tree-Based' if model_name in tree_names else 'Classical'

def compute_macro_f1(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    f1_vals = []
    for cls_name, cls_metrics in report.items():
        if cls_name in ['accuracy', 'macro avg', 'weighted avg']:
            continue
        if isinstance(cls_metrics, dict) and 'f1-score' in cls_metrics:
            f1_vals.append(cls_metrics['f1-score'])
    return float(np.mean(f1_vals)) if len(f1_vals) > 0 else np.nan

def get_pickle_model_size_bytes(model_obj):
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as tmp_file:
        joblib.dump(model_obj, tmp_file.name)
        tmp_path = tmp_file.name
    size_bytes = os.path.getsize(tmp_path)
    os.remove(tmp_path)
    return size_bytes

def evaluate_sklearn_model(model_obj, X_test, y_test, model_name: str, model_size_bytes: int):
    start = time.time()
    y_pred = model_obj.predict(X_test)
    infer_time_s = time.time() - start

    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    return {
        'Model': model_name,
        'Family': get_model_family(model_name),
        'Accuracy (%)': accuracy_score(y_test, y_pred) * 100,
        'Weighted F1 (%)': f * 100,
        'Macro F1 (%)': compute_macro_f1(y_test, y_pred) * 100,
        'Precision (%)': p * 100,
        'Recall (%)': r * 100,
        'Inference Time (ms/sample)': (infer_time_s / len(X_test)) * 1000,
        'Model Size (bytes)': model_size_bytes,
    }

def make_result_row(dataset_name, technique_name, baseline_name, optimized_name, baseline_metrics, optimized_metrics, note=''):
    return {
        'Dataset': dataset_name,
        'Technique': technique_name,
        'Baseline Model': baseline_name,
        'Optimized Model': optimized_name,
        'Baseline Accuracy (%)': baseline_metrics['Accuracy (%)'],
        'Optimized Accuracy (%)': optimized_metrics['Accuracy (%)'],
        'Baseline Weighted F1 (%)': baseline_metrics['Weighted F1 (%)'],
        'Optimized Weighted F1 (%)': optimized_metrics['Weighted F1 (%)'],
        'Baseline Macro F1 (%)': baseline_metrics['Macro F1 (%)'],
        'Optimized Macro F1 (%)': optimized_metrics['Macro F1 (%)'],
        'Baseline Inference Time (ms/sample)': baseline_metrics['Inference Time (ms/sample)'],
        'Optimized Inference Time (ms/sample)': optimized_metrics['Inference Time (ms/sample)'],
        'Baseline Model Size (bytes)': baseline_metrics['Model Size (bytes)'],
        'Optimized Model Size (bytes)': optimized_metrics['Model Size (bytes)'],
        'Weighted F1 Change (%)': optimized_metrics['Weighted F1 (%)'] - baseline_metrics['Weighted F1 (%)'],
        'Macro F1 Change (%)': optimized_metrics['Macro F1 (%)'] - baseline_metrics['Macro F1 (%)'],
        'Inference Time Change (%)': ((optimized_metrics['Inference Time (ms/sample)'] - baseline_metrics['Inference Time (ms/sample)']) / max(baseline_metrics['Inference Time (ms/sample)'], 1e-9)) * 100,
        'Model Size Change (%)': ((optimized_metrics['Model Size (bytes)'] - baseline_metrics['Model Size (bytes)']) / max(baseline_metrics['Model Size (bytes)'], 1e-9)) * 100,
        'Note': note,
    }

def _plot_change_bars(result_df, value_col, title_text):
    plot_df = result_df.copy().sort_values(value_col, ascending=True)
    fig, ax = plt.subplots(figsize=(14, max(6, len(plot_df) * 0.35)))
    bars = ax.barh(plot_df['Optimized Model'], plot_df[value_col], color=sns.color_palette('muted', len(plot_df)))
    for bar, val in zip(bars, plot_df[value_col]):
        if pd.isna(val):
            continue
        x_pos = val + 0.05 if val >= 0 else val - 0.8
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2, f'{val:.2f}%', va='center', fontsize=8)
    ax.set_xlabel(value_col)
    ax.set_ylabel('Optimized Model')
    ax.set_title(title_text)
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

def _plot_baseline_vs_optimized(result_df, metric_baseline_col, metric_optimized_col, title_text):
    labels = result_df['Optimized Model'].tolist()
    baseline_vals = result_df[metric_baseline_col].tolist()
    optimized_vals = result_df[metric_optimized_col].tolist()
    y = np.arange(len(labels))
    height = 0.35
    fig, ax = plt.subplots(figsize=(14, max(6, len(labels) * 0.35)))
    bars1 = ax.barh(y - height / 2, baseline_vals, height, label='Baseline', color=sns.color_palette('deep')[0])
    bars2 = ax.barh(y + height / 2, optimized_vals, height, label='Optimized', color=sns.color_palette('deep')[1])
    for bar, val in zip(bars1, baseline_vals):
        if pd.isna(val):
            continue
        ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)
    for bar, val in zip(bars2, optimized_vals):
        if pd.isna(val):
            continue
        ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Value')
    ax.set_ylabel('Model')
    ax.set_title(title_text)
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()

def _plot_cost_vs_f1(result_df, title_text):
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(data=result_df, x='Optimized Inference Time (ms/sample)', y='Optimized Weighted F1 (%)', hue='Technique', s=120, ax=ax)
    for _, row in result_df.iterrows():
        if pd.isna(row['Optimized Inference Time (ms/sample)']) or pd.isna(row['Optimized Weighted F1 (%)']):
            continue
        ax.text(row['Optimized Inference Time (ms/sample)'], row['Optimized Weighted F1 (%)'], row['Optimized Model'], fontsize=7, ha='left', va='bottom')
    ax.set_xlabel('Optimized Inference Time (ms/sample)')
    ax.set_ylabel('Optimized Weighted F1 (%)')
    ax.set_title(title_text)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
def train_classical_variant(model_name, baseline_model, X_train, y_train, technique_name):
    lower_name = model_name.lower()

    if technique_name == 'Feature Selection':
        selector = SelectKBest(score_func=mutual_info_classif, k=min(32, X_train.shape[1]))
        if 'logistic regression' in lower_name:
            estimator = LogisticRegression(max_iter=1000, n_jobs=-1)
        elif 'svm' in lower_name:
            estimator = SVC(kernel='linear')
        elif 'knn' in lower_name:
            estimator = KNeighborsClassifier(n_neighbors=5, weights='distance')
        else:
            return None, None, 'Not finished due unsupported classical model.'
        variant = Pipeline([('selector', selector), ('estimator', estimator)])
        variant.fit(X_train, y_train)
        return variant, None, ''

    elif technique_name == 'PCA':
        n_comp = min(32, max(2, min(X_train.shape[0] - 1, X_train.shape[1])))
        if 'logistic regression' in lower_name:
            estimator = LogisticRegression(max_iter=1000, n_jobs=-1)
        elif 'svm' in lower_name:
            estimator = SVC(kernel='linear')
        elif 'knn' in lower_name:
            estimator = KNeighborsClassifier(n_neighbors=5, weights='distance')
        else:
            return None, None, 'Not finished due unsupported classical model.'
        variant = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=n_comp, random_state=RANDOM_STATE)), ('estimator', estimator)])
        variant.fit(X_train, y_train)
        return variant, None, ''

    elif technique_name == 'L1 Sparsification':
        if 'logistic regression' in lower_name:
            variant = Pipeline([('scaler', StandardScaler()), ('estimator', LogisticRegression(penalty='l1', solver='saga', C=0.5, max_iter=1500, n_jobs=-1))])
            variant.fit(X_train, y_train)
            return variant, None, ''
        else:
            return None, None, 'Not finished due L1 sparsification being implemented only for Logistic Regression in this notebook.'

    elif technique_name == 'Prototype Reduction':
        if 'knn' in lower_name:
            train_size = max(10, min(int(len(X_train) * 0.5), len(X_train)-1))
            X_sub, _, y_sub, _ = train_test_split(X_train, y_train, train_size=train_size, stratify=y_train, random_state=RANDOM_STATE)
            variant = KNeighborsClassifier(n_neighbors=max(3, min(5, baseline_model.get_params().get('n_neighbors', 5))), weights=baseline_model.get_params().get('weights', 'uniform'))
            variant.fit(X_sub, y_sub)
            return variant, {'prototype_rows': len(X_sub)}, ''
        else:
            return None, None, 'Not finished due prototype reduction being applicable only to KNN in this notebook.'

    else:
        return None, None, 'Not finished due unknown technique.'

## Baseline Metrics

In [ ]:
# CICEVSE2024

baseline_rows_cicevse = []

for model_name, model_obj in [
    ('Logistic Regression', lr_cicevse_model),
    ('SVM', svm_cicevse_model),
    ('KNN', knn_cicevse_model)
]:
    model_size_bytes = get_pickle_model_size_bytes(model_obj)
    metrics = evaluate_sklearn_model(model_obj, X_test_cicevse, y_test_cicevse, model_name, model_size_bytes=model_size_bytes)
    baseline_rows_cicevse.append(metrics)

baseline_df_cicevse = pd.DataFrame(baseline_rows_cicevse)
display(baseline_df_cicevse)

In [ ]:
# CICIDS2017

baseline_rows_cicids = []

for model_name, model_obj in [
    ('Logistic Regression', lr_cicids_model),
    ('SVM', svm_cicids_model),
    ('KNN', knn_cicids_model)
]:
    model_size_bytes = get_pickle_model_size_bytes(model_obj)
    metrics = evaluate_sklearn_model(model_obj, X_test_cicids, y_test_cicids, model_name, model_size_bytes=model_size_bytes)
    baseline_rows_cicids.append(metrics)

baseline_df_cicids = pd.DataFrame(baseline_rows_cicids)
display(baseline_df_cicids)

In [ ]:
# NF-ToN-IoT-V2

baseline_rows_nfton = []

for model_name, model_obj in [
    ('Logistic Regression', lr_nfton_model),
    ('SVM', svm_nfton_model),
    ('KNN', knn_nfton_model)
]:
    model_size_bytes = get_pickle_model_size_bytes(model_obj)
    metrics = evaluate_sklearn_model(model_obj, X_test_nfton, y_test_nfton, model_name, model_size_bytes=model_size_bytes)
    baseline_rows_nfton.append(metrics)

baseline_df_nfton = pd.DataFrame(baseline_rows_nfton)
display(baseline_df_nfton)

## Feature Selection

In [ ]:
# Feature Selection — CICEVSE2024

feature_selection_rows_cicevse = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicevse_model),
    ('SVM', svm_cicevse_model),
    ('KNN', knn_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Feature Selection')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Feature Selection]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Feature Selection]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    feature_selection_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Feature Selection', model_name, f'{model_name} [Feature Selection]', baseline_metrics, optimized_metrics, note=note)
    )

feature_selection_df_cicevse = pd.DataFrame(feature_selection_rows_cicevse)
display(feature_selection_df_cicevse)

In [ ]:
# Feature Selection — CICIDS2017

feature_selection_rows_cicids = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicids_model),
    ('SVM', svm_cicids_model),
    ('KNN', knn_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Feature Selection')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Feature Selection]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Feature Selection]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    feature_selection_rows_cicids.append(
        make_result_row('CICIDS2017', 'Feature Selection', model_name, f'{model_name} [Feature Selection]', baseline_metrics, optimized_metrics, note=note)
    )

feature_selection_df_cicids = pd.DataFrame(feature_selection_rows_cicids)
display(feature_selection_df_cicids)

In [ ]:
# Feature Selection — NF-ToN-IoT-V2

feature_selection_rows_nfton = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_nfton_model),
    ('SVM', svm_nfton_model),
    ('KNN', knn_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Feature Selection')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Feature Selection]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Feature Selection]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    feature_selection_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Feature Selection', model_name, f'{model_name} [Feature Selection]', baseline_metrics, optimized_metrics, note=note)
    )

feature_selection_df_nfton = pd.DataFrame(feature_selection_rows_nfton)
display(feature_selection_df_nfton)

## PCA

In [ ]:
# PCA — CICEVSE2024

pca_rows_cicevse = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicevse_model),
    ('SVM', svm_cicevse_model),
    ('KNN', knn_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'PCA')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [PCA]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [PCA]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    pca_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'PCA', model_name, f'{model_name} [PCA]', baseline_metrics, optimized_metrics, note=note)
    )

pca_df_cicevse = pd.DataFrame(pca_rows_cicevse)
display(pca_df_cicevse)

In [ ]:
# PCA — CICIDS2017

pca_rows_cicids = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicids_model),
    ('SVM', svm_cicids_model),
    ('KNN', knn_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'PCA')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [PCA]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [PCA]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    pca_rows_cicids.append(
        make_result_row('CICIDS2017', 'PCA', model_name, f'{model_name} [PCA]', baseline_metrics, optimized_metrics, note=note)
    )

pca_df_cicids = pd.DataFrame(pca_rows_cicids)
display(pca_df_cicids)

In [ ]:
# PCA — NF-ToN-IoT-V2

pca_rows_nfton = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_nfton_model),
    ('SVM', svm_nfton_model),
    ('KNN', knn_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'PCA')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [PCA]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [PCA]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    pca_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'PCA', model_name, f'{model_name} [PCA]', baseline_metrics, optimized_metrics, note=note)
    )

pca_df_nfton = pd.DataFrame(pca_rows_nfton)
display(pca_df_nfton)

## L1 Sparsification

In [ ]:
# L1 Sparsification — CICEVSE2024

l1_sparsification_rows_cicevse = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicevse_model),
    ('SVM', svm_cicevse_model),
    ('KNN', knn_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'L1 Sparsification')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [L1 Sparsification]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [L1 Sparsification]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    l1_sparsification_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'L1 Sparsification', model_name, f'{model_name} [L1 Sparsification]', baseline_metrics, optimized_metrics, note=note)
    )

l1_sparsification_df_cicevse = pd.DataFrame(l1_sparsification_rows_cicevse)
display(l1_sparsification_df_cicevse)

In [ ]:
# L1 Sparsification — CICIDS2017

l1_sparsification_rows_cicids = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicids_model),
    ('SVM', svm_cicids_model),
    ('KNN', knn_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'L1 Sparsification')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [L1 Sparsification]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [L1 Sparsification]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    l1_sparsification_rows_cicids.append(
        make_result_row('CICIDS2017', 'L1 Sparsification', model_name, f'{model_name} [L1 Sparsification]', baseline_metrics, optimized_metrics, note=note)
    )

l1_sparsification_df_cicids = pd.DataFrame(l1_sparsification_rows_cicids)
display(l1_sparsification_df_cicids)

In [ ]:
# L1 Sparsification — NF-ToN-IoT-V2

l1_sparsification_rows_nfton = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_nfton_model),
    ('SVM', svm_nfton_model),
    ('KNN', knn_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'L1 Sparsification')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [L1 Sparsification]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [L1 Sparsification]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    l1_sparsification_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'L1 Sparsification', model_name, f'{model_name} [L1 Sparsification]', baseline_metrics, optimized_metrics, note=note)
    )

l1_sparsification_df_nfton = pd.DataFrame(l1_sparsification_rows_nfton)
display(l1_sparsification_df_nfton)

## Prototype Reduction

In [ ]:
# Prototype Reduction — CICEVSE2024

prototype_reduction_rows_cicevse = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicevse_model),
    ('SVM', svm_cicevse_model),
    ('KNN', knn_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Prototype Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Prototype Reduction]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Prototype Reduction]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    prototype_reduction_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Prototype Reduction', model_name, f'{model_name} [Prototype Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

prototype_reduction_df_cicevse = pd.DataFrame(prototype_reduction_rows_cicevse)
display(prototype_reduction_df_cicevse)

In [ ]:
# Prototype Reduction — CICIDS2017

prototype_reduction_rows_cicids = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_cicids_model),
    ('SVM', svm_cicids_model),
    ('KNN', knn_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Prototype Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Prototype Reduction]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Prototype Reduction]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    prototype_reduction_rows_cicids.append(
        make_result_row('CICIDS2017', 'Prototype Reduction', model_name, f'{model_name} [Prototype Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

prototype_reduction_df_cicids = pd.DataFrame(prototype_reduction_rows_cicids)
display(prototype_reduction_df_cicids)

In [ ]:
# Prototype Reduction — NF-ToN-IoT-V2

prototype_reduction_rows_nfton = []

for model_name, baseline_model in [
    ('Logistic Regression', lr_nfton_model),
    ('SVM', svm_nfton_model),
    ('KNN', knn_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, extra_info, note = train_classical_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Prototype Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Prototype Reduction]',
            'Family': 'Classical',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Prototype Reduction]', model_size_bytes=variant_size_bytes)

    if extra_info is not None and 'prototype_rows' in extra_info:
        note = (note + ' ' + f"Prototype subset rows used: {extra_info['prototype_rows']}.").strip()

    prototype_reduction_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Prototype Reduction', model_name, f'{model_name} [Prototype Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

prototype_reduction_df_nfton = pd.DataFrame(prototype_reduction_rows_nfton)
display(prototype_reduction_df_nfton)

## Combined Task 1 Result Table

In [ ]:
classical_all_df = pd.concat([
    feature_selection_df_cicevse, feature_selection_df_cicids, feature_selection_df_nfton,
    pca_df_cicevse, pca_df_cicids, pca_df_nfton,
    l1_sparsification_df_cicevse, l1_sparsification_df_cicids, l1_sparsification_df_nfton,
    prototype_reduction_df_cicevse, prototype_reduction_df_cicids, prototype_reduction_df_nfton,
], ignore_index=True)

display(classical_all_df)

## Final Task 1 Plots

In [ ]:
_plot_change_bars(classical_all_df, 'Weighted F1 Change (%)', 'Classical Optimization — Weighted F1 Change')
_plot_change_bars(classical_all_df, 'Macro F1 Change (%)', 'Classical Optimization — Macro F1 Change')
_plot_change_bars(classical_all_df, 'Inference Time Change (%)', 'Classical Optimization — Inference Time Change')
_plot_change_bars(classical_all_df, 'Model Size Change (%)', 'Classical Optimization — Model Size Change')

_plot_baseline_vs_optimized(classical_all_df, 'Baseline Weighted F1 (%)', 'Optimized Weighted F1 (%)', 'Classical Optimization — Baseline vs Optimized Weighted F1')
_plot_baseline_vs_optimized(classical_all_df, 'Baseline Inference Time (ms/sample)', 'Optimized Inference Time (ms/sample)', 'Classical Optimization — Baseline vs Optimized Inference Time')
_plot_cost_vs_f1(classical_all_df, 'Classical Optimization — Optimized Weighted F1 vs Inference Time')